In [17]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [1]:
#!/usr/bin/env python3
# ============================================================
# Mobile NMF — ORIGINAL vs Injection comparison (per category)
# Correct metric:
#   Avg number of TARGET-category apps per user in Top-K
# Uses ORIGINAL schema: user_id, app_id, est_score, rank, app_root
# Python 3.8+
# ============================================================

from pathlib import Path
from typing import Dict, List
import re
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ===================== PATHS =====================
ROOT = Path("/home/moshtasa/Research/phd-svd-recsys/Mobile/NMF/NMF")
ORIG_DIR = ROOT
OUT_BASE = ROOT / "figures"

K_LIST = [15, 25, 35]
N_LIST = [27, 105, 2643]

# ===================== CATEGORIES (filename keys) =====================
CATEGORIES = [
    "Business___Finance",
    "Media___Creativity",
    "Games",
    "Medical",
    "Health___Lifestyle",
    "Productivity___Tools",
    "Social___Comms",
    "Info___Reading",
    "Learning___Education",
    "Travel___Navigation",
]

# Map filename token -> ORIGINAL "app_root" value
CAT_TO_APP_ROOT = {
    "Business___Finance": "Business & Finance",
    "Media___Creativity": "Media & Creativity",
    "Games": "Games",
    "Medical": "Medical",
    "Health___Lifestyle": "Health & Lifestyle",
    "Productivity___Tools": "Productivity & Tools",
    "Social___Comms": "Social & Comms",
    "Info___Reading": "Info & Reading",
    "Learning___Education": "Learning & Education",
    "Travel___Navigation": "Travel & Navigation",
}

# ===================== HELPERS =====================
def load_csv(p: Path) -> pd.DataFrame:
    df = pd.read_csv(p, low_memory=False)
    for c in ["user_id", "app_id", "rank", "est_score", "app_root"]:
        if c not in df.columns:
            df[c] = pd.NA
    return df


def norm_app_root(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("&", "and")
    s = re.sub(r"[\s_]+", "", s)
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s


def filter_to_target(df: pd.DataFrame, cat_key: str) -> pd.DataFrame:
    target_root = CAT_TO_APP_ROOT.get(cat_key, cat_key.replace("___", " "))
    tnorm = norm_app_root(target_root)
    return df[df["app_root"].apply(norm_app_root) == tnorm]


def avg_target_apps_per_user(df_target: pd.DataFrame) -> float:
    if df_target.empty:
        return 0.0
    return float(df_target.groupby("user_id").size().mean())


def per_app_ranking(df_target: pd.DataFrame) -> pd.DataFrame:
    if df_target.empty:
        return pd.DataFrame(columns=[
            "rank", "app_id", "freq", "users_n",
            "avg_rank", "avg_est_score", "rank1_count"
        ])

    g = df_target.groupby("app_id", as_index=False)
    out = g.agg(
        freq=("app_id", "size"),
        users_n=("user_id", "nunique"),
        avg_rank=("rank", "mean"),
        avg_est_score=("est_score", "mean"),
        rank1_count=("rank", lambda s: int((s == 1).sum())),
    )

    out = out.sort_values(
        ["freq", "rank1_count", "avg_rank"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    out.insert(0, "rank", out.index + 1)
    return out

# ===================== PLOT =====================
def plot_category(cat_key: str,
                  data_by_k: Dict[int, Dict[str, float]],
                  out_png: Path):

    ks = sorted(data_by_k.keys())
    groups = ["Original"] + [str(n) for n in N_LIST]

    width = 0.8 / len(groups)
    x = list(range(len(ks)))

    plt.figure(figsize=(9, 4), dpi=160)

    for i, g in enumerate(groups):
        vals = [data_by_k[k].get(g, 0.0) for k in ks]
        offs = [j + (i - len(groups)/2) * width for j in x]
        plt.bar(offs, vals, width=width, label=g)

    plt.xticks(x, [f"Top-{k}" for k in ks])
    plt.ylabel("Avg # TARGET-category apps per user")

    display_name = CAT_TO_APP_ROOT.get(cat_key, cat_key.replace("___", " "))
    plt.title(f"NMF – {display_name}")

    plt.legend()
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png)
    plt.close()

# ===================== MAIN PER CATEGORY =====================
def process_category(cat_key: str):
    OUT_DIR = OUT_BASE / f"{cat_key}_explanation"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    display_name = CAT_TO_APP_ROOT.get(cat_key, cat_key.replace("___", " "))

    print("\n" + "=" * 80)
    print(f"CATEGORY: {cat_key}  (app_root='{display_name}')")
    print("=" * 80)

    text_lines = [f"{cat_key.lower()}:"]
    all_tables: List[pd.DataFrame] = []
    data_by_k: Dict[int, Dict[str, float]] = {k: {} for k in K_LIST}

    # ---------- ORIGINAL ----------
    print("\n[ORIGINAL]")
    for K in K_LIST:
        fp = ORIG_DIR / f"ORIGINAL_{K}recommendation.csv"
        if not fp.exists():
            print(f"  Top-{K}: MISSING")
            continue

        df = load_csv(fp)
        df_target = filter_to_target(df, cat_key)

        uniq_apps = df_target["app_id"].nunique()
        avg_target = avg_target_apps_per_user(df_target)

        data_by_k[K]["Original"] = avg_target
        text_lines.append(f"original {K}: number_of_unique_apps: {uniq_apps}")

        print(f"  Top-{K}: unique_apps={uniq_apps}, avg_target_apps_per_user={avg_target:.4f}")

        ranked = per_app_ranking(df_target)
        ranked["dataset"] = f"original_{K}"
        ranked["category_key"] = cat_key
        ranked["app_root"] = display_name
        all_tables.append(ranked)

    # ---------- INJECTIONS ----------
    for N in N_LIST:
        print(f"\n[{N}u INJECTION]")
        for K in K_LIST:
            fp = ROOT / f"f_{cat_key}_{N}u_pos5_neg1_all_{K}recommendation.csv"
            if not fp.exists():
                print(f"  Top-{K}: MISSING")
                continue

            df = load_csv(fp)

            # Injection files are category-specific, but filter safely
            df_target = filter_to_target(df, cat_key) if "app_root" in df.columns else df

            uniq_apps = df_target["app_id"].nunique()
            avg_target = avg_target_apps_per_user(df_target)

            data_by_k[K][str(N)] = avg_target
            text_lines.append(f"{N}u, {K}, number_of_unique_apps: {uniq_apps}")

            print(f"  Top-{K}: unique_apps={uniq_apps}, avg_target_apps_per_user={avg_target:.4f}")

            ranked = per_app_ranking(df_target)
            ranked["dataset"] = f"{N}u_{K}"
            ranked["category_key"] = cat_key
            ranked["app_root"] = display_name
            all_tables.append(ranked)

    # ---------- SAVE ----------
    with open(OUT_DIR / "explanation.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(text_lines))

    if all_tables:
        pd.concat(all_tables).to_csv(
            OUT_DIR / "per_app_ranking.csv",
            index=False
        )

    plot_category(
        cat_key,
        data_by_k,
        OUT_DIR / f"{cat_key.lower()}_pos5.png"
    )

    print(f"\n[OK] Saved → {OUT_DIR}")

# ===================== ENTRY =====================
def main():
    for cat_key in CATEGORIES:
        process_category(cat_key)

if __name__ == "__main__":
    main()



CATEGORY: Business___Finance  (app_root='Business & Finance')

[ORIGINAL]
  Top-15: unique_apps=135, avg_target_apps_per_user=1.3844
  Top-25: unique_apps=186, avg_target_apps_per_user=1.7064
  Top-35: unique_apps=229, avg_target_apps_per_user=2.0520

[27u INJECTION]
  Top-15: unique_apps=203, avg_target_apps_per_user=9.9041
  Top-25: unique_apps=262, avg_target_apps_per_user=14.3748
  Top-35: unique_apps=310, avg_target_apps_per_user=17.9807

[105u INJECTION]
  Top-15: unique_apps=349, avg_target_apps_per_user=10.0460


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.